# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Sun Mar 29 11:45:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   24C    P8              1W /  250W |     502MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any
from utils import make_loaders, train, validate, fit, evaluate

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(1)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [3]:
LOG_WANDB = False # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Data Loading

In [4]:
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 20% of training set used for validation

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

classes: list[str] = train_set.classes
print("Classes:", classes)

Dataset split — train: 40,000  val: 10,000  test: 10,000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

| Layer | Detail |
|---|---|
| Conv Block 1 | Conv2d(3 → 32, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 2 | Conv2d(32 → 64, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| Conv Block 3 | Conv2d(64 → 128, 3×3, pad 1) → LeakyReLU → MaxPool2d |
| FC 1 | Linear(128 × 4 × 4 → 256) → LeakyReLU |
| FC 2 | Linear(256 → `num_classes`) |

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(
        self,
        num_classes: int = 10,
        activation: nn.Module = nn.LeakyReLU(),
    ) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,  32,  kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            activation,
            nn.MaxPool2d(2, 2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            activation,
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [6]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(
    model=model_a, 
    optimizer=optimizer_a,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch  1/50 | train loss 2.2984, train acc 13.95% | val loss 2.2937, val acc 18.08%
Epoch  2/50 | train loss 2.2865, train acc 16.89% | val loss 2.2764, val acc 18.58%
Epoch  3/50 | train loss 2.2535, train acc 20.69% | val loss 2.2164, val acc 22.95%
Epoch  4/50 | train loss 2.1565, train acc 23.35% | val loss 2.0957, val acc 25.52%
Epoch  5/50 | train loss 2.0504, train acc 26.84% | val loss 2.0068, val acc 29.37%
Epoch  6/50 | train loss 1.9575, train acc 30.66% | val loss 1.9097, val acc 31.90%
Epoch  7/50 | train loss 1.8661, train acc 33.47% | val loss 1.8407, val acc 33.21%
Epoch  8/50 | train loss 1.7980, train acc 35.67% | val loss 1.7864, val acc 34.68%
Epoch  9/50 | train loss 1.7431, train acc 37.24% | val loss 1.7771, val acc 34.66%
Epoch 10/50 | train loss 1.7054, train acc 38.78% | val loss 1.7322, val acc 37.10%
Epoch 11/50 | train loss 1.6607, train acc 40.44% | val loss 1.6662, val acc 39.67%
Epoch 12/50 | train loss 1.6233, train acc 41.64% | val loss 1.6470, val acc

In [7]:
_ = evaluate(model=model_a, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="SGD + LeakyReLU")

[SGD + LeakyReLU] Test loss: 1.0594 | Test acc: 63.23%


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(
    model=model_b, 
    optimizer=optimizer_b,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch  1/50 | train loss 1.5991, train acc 41.71% | val loss 1.3407, val acc 51.28%
Epoch  2/50 | train loss 1.2005, train acc 57.06% | val loss 1.1572, val acc 58.32%
Epoch  3/50 | train loss 0.9974, train acc 64.44% | val loss 0.9858, val acc 65.33%
Epoch  4/50 | train loss 0.8653, train acc 69.47% | val loss 0.9639, val acc 66.17%
Epoch  5/50 | train loss 0.7639, train acc 73.29% | val loss 0.8597, val acc 70.31%
Epoch  6/50 | train loss 0.6752, train acc 76.59% | val loss 0.8249, val acc 71.05%
Epoch  7/50 | train loss 0.5918, train acc 79.61% | val loss 0.8683, val acc 70.11%
Epoch  8/50 | train loss 0.5302, train acc 81.57% | val loss 0.8364, val acc 72.29%
Epoch  9/50 | train loss 0.4460, train acc 84.47% | val loss 0.8346, val acc 73.06%
Epoch 10/50 | train loss 0.3700, train acc 87.43% | val loss 0.9609, val acc 70.68%
Epoch 11/50 | train loss 0.3150, train acc 89.24% | val loss 0.9076, val acc 72.77%
Epoch 12/50 | train loss 0.2452, train acc 91.63% | val loss 0.9205, val acc

In [9]:
_ = evaluate(model=model_b, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + LeakyReLU")

[Adam + LeakyReLU] Test loss: 0.8246 | Test acc: 71.14%


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [10]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(
    model=model_c, 
    optimizer=optimizer_c,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

Epoch  1/50 | train loss 1.4624, train acc 47.64% | val loss 1.2718, val acc 54.04%
Epoch  2/50 | train loss 1.0717, train acc 62.12% | val loss 0.9983, val acc 64.51%
Epoch  3/50 | train loss 0.8844, train acc 68.81% | val loss 0.9518, val acc 66.53%
Epoch  4/50 | train loss 0.7733, train acc 73.18% | val loss 0.8537, val acc 70.54%
Epoch  5/50 | train loss 0.6588, train acc 77.36% | val loss 0.8639, val acc 70.11%
Epoch  6/50 | train loss 0.5622, train acc 80.93% | val loss 0.8203, val acc 72.06%
Epoch  7/50 | train loss 0.4681, train acc 84.64% | val loss 0.8270, val acc 72.72%
Epoch  8/50 | train loss 0.3799, train acc 87.81% | val loss 0.8151, val acc 73.27%
Epoch  9/50 | train loss 0.2999, train acc 90.95% | val loss 0.8399, val acc 72.84%
Epoch 10/50 | train loss 0.2230, train acc 93.82% | val loss 0.8593, val acc 72.82%
Epoch 11/50 | train loss 0.1600, train acc 96.20% | val loss 0.8983, val acc 72.94%
Epoch 12/50 | train loss 0.1060, train acc 98.22% | val loss 0.9265, val acc

In [11]:
_ = evaluate(model=model_c, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + Tanh")

[Adam + Tanh] Test loss: 0.8081 | Test acc: 73.20%
